# <img align="left" src="./images/movie_camera.png"     style=" width:40px;  " > 实践实验：协同过滤推荐系统

在本练习中，你将实现协同过滤，来为电影构建一个推荐系统。

# <img align="left" src="./images/film_reel.png"     style=" width:40px;  " > 大纲
- [ 1 - 记号](#1)
- [ 2 - 推荐系统](#2)
- [ 3 - 电影评分数据集](#3)
- [ 4 - 协同过滤学习算法](#4)
  - [ 4.1 协同过滤代价函数](#4.1)
    - [ 练习 1](#ex01)
- [ 5 - 学习电影推荐](#5)
- [ 6 - 推荐](#6)
- [ 7 - 恭喜完成！](#7)





_**注意：** 为防止自动评分器报错，你不得编辑或删除本实验中不参与评分的单元格。也请不要添加任何新单元格。
**一旦你通过了本次作业** 并想试验任何不参与评分的代码，你可以按照本笔记本底部的说明操作。_

## 软件包 <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;   " >
我们将使用现在已经熟悉的 NumPy 和 Tensorflow 软件包。

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from recsys_utils import *

<a name="1"></a>
## 1 - 记号


| 通用 <br />  记号 | 描述 | Python（如有） |
| :--- | :--- | :--- |
| $r(i,j)$ | 标量；如果用户 j 给电影 i 评过分则 = 1，否则 = 0 |  |
| $y(i,j)$ | 标量；= 用户 j 给电影 i 的评分（如果 r(i,j) = 1 则有定义） |  |
| $\mathbf{w}^{(j)}$ | 向量；用户 j 的参数 |  |
| $b^{(j)}$ | 标量；用户 j 的参数 |  |
| $\mathbf{x}^{(i)}$ | 向量；电影 i 的特征评分 |  |
| $n_u$ | 用户数量 | num_users |
| $n_m$ | 电影数量 | num_movies |
| $n$ | 特征数量 | num_features |
| $\mathbf{X}$ | 由向量 $\mathbf{x}^{(i)}$ 组成的矩阵 | X |
| $\mathbf{W}$ | 由向量 $\mathbf{w}^{(j)}$ 组成的矩阵 | W |
| $\mathbf{b}$ | 由偏置参数 $b^{(j)}$ 组成的向量 | b |
| $\mathbf{R}$ | 由元素 $r(i,j)$ 组成的矩阵 | R |



<a name="2"></a>
## 2 - 推荐系统 <img align="left" src="./images/film_rating.png" style=" width:40px;  " >
在本实验中，你将实现协同过滤学习算法，并把它应用到一个电影评分数据集上。
协同过滤推荐系统的目标是生成两个向量：对每个用户，生成一个体现该用户电影品味的"参数向量"。对每部电影，生成一个同样大小的特征向量，它体现了对这部电影的某种描述。这两个向量的点积加上偏置项，应产生对该用户可能给那部电影的评分的一个估计。

下面的图详细说明了这些向量是如何学习得到的。

<figure>
   <img src="./images/ColabFilterLearn.PNG"  style="width:740px;height:250px;" >
</figure>

已有的评分以矩阵形式提供，如图所示。$Y$ 包含评分；从 0.5 到 5（含），以 0.5 为步长。如果电影未被评分则为 0。$R$ 在电影被评分处为 1。电影在行中，用户在列中。每个用户有一个参数向量 $w^{user}$ 和偏置。每部电影有一个特征向量 $x^{movie}$。这些向量通过使用已有的用户/电影评分作为训练数据而被同时学习。上面展示了一个训练样本：$\mathbf{w}^{(1)} \cdot \mathbf{x}^{(1)} + b^{(1)} = 4$。值得注意的是，特征向量 $x^{movie}$ 必须满足所有用户，而用户向量 $w^{user}$ 必须满足所有电影。这就是这种方法名称的来源——所有用户协同生成评分集。

<figure>
   <img src="./images/ColabFilterUse.PNG"  style="width:640px;height:250px;" >
</figure>

一旦特征向量和参数被学习好，它们就可以用来预测一个用户可能如何给一部未评分的电影打分。这在上面的图中有所展示。该方程是为用户一预测电影零评分的一个例子。


在本练习中，你将实现函数 `cofiCostFunc`，它计算协同过滤的目标函数。实现目标函数之后，你将使用一个 TensorFlow 自定义训练循环来学习协同过滤的参数。第一步是详细说明本实验中将使用的数据集和数据结构。

<a name="3"></a>
## 3 - 电影评分数据集 <img align="left" src="./images/film_rating.png"     style=" width:40px;  " >
该数据集源自 [MovieLens "ml-latest-small"](https://grouplens.org/datasets/movielens/latest/) 数据集。
[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集有 600 位用户评分的 9000 部电影。该数据集在规模上已被缩减，以聚焦于 2000 年以来这些年份的电影。这个数据集由以 0.5 到 5、步长为 0.5 的评分构成。缩减后的数据集有 $n_u = 443$ 位用户，和 $n_m= 4778$ 部电影。

在下面，你将把电影数据集加载到变量 $Y$ 和 $R$ 中。

矩阵 $Y$（一个 $n_m \times n_u$ 矩阵）存储评分 $y^{(i,j)}$。矩阵 $R$ 是一个二元取值的指示矩阵，其中如果用户 $j$ 给电影 $i$ 评过分则 $R(i,j) = 1$，否则 $R(i,j)=0$。

在本练习的这一部分中，你还将使用矩阵 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$：

$$\mathbf{X} = 
\begin{bmatrix}
--- (\mathbf{x}^{(0)})^T --- \\
--- (\mathbf{x}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{x}^{(n_m-1)})^T --- \\
\end{bmatrix} , \quad
\mathbf{W} = 
\begin{bmatrix}
--- (\mathbf{w}^{(0)})^T --- \\
--- (\mathbf{w}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{w}^{(n_u-1)})^T --- \\
\end{bmatrix},\quad
\mathbf{ b} = 
\begin{bmatrix}
 b^{(0)}  \\
 b^{(1)} \\
\vdots \\
b^{(n_u-1)} \\
\end{bmatrix}\quad
$$ 

$\mathbf{X}$ 的第 $i$ 行对应第 $i$ 部电影的特征向量 $x^{(i)}$，$\mathbf{W}$ 的第 $j$ 行对应第 $j$ 位用户的一个参数向量 $\mathbf{w}^{(j)}$。$x^{(i)}$ 和 $\mathbf{w}^{(j)}$ 都是 $n$ 维向量。就本练习而言，你将使用 $n=10$，因此 $\mathbf{x}^{(i)}$ 和 $\mathbf{w}^{(j)}$ 有 10 个元素。相应地，$\mathbf{X}$ 是一个 $n_m \times 10$ 矩阵，$\mathbf{W}$ 是一个 $n_u \times 10$ 矩阵。

我们将从加载电影评分数据集开始，以理解数据的结构。
我们将用电影数据集加载 $Y$ 和 $R$。
我们还将用预先计算好的值加载 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$。这些值将在实验稍后被学习，但我们会用预先计算好的值来开发代价模型。

In [2]:
#Load data
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y, R = load_ratings_small()

print("Y", Y.shape, "R", R.shape)
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("num_features", num_features)
print("num_movies",   num_movies)
print("num_users",    num_users)

Y (4778, 443) R (4778, 443)
X (4778, 10)
W (443, 10)
b (1, 443)
num_features 10
num_movies 4778
num_users 443


In [3]:
#  From the matrix, we can compute statistics like average rating.
tsmean =  np.mean(Y[0, R[0, :].astype(bool)])
print(f"Average rating for movie 1 : {tsmean:0.3f} / 5" )

Average rating for movie 1 : 3.400 / 5


<a name="4"></a>
## 4 - 协同过滤学习算法 <img align="left" src="./images/film_filter.png"     style=" width:40px;  " >

现在，你将开始实现协同过滤学习算法。你将从实现目标函数开始。

在电影推荐这一设定下的协同过滤算法，考虑一组 $n$ 维参数向量 $\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)}$、$\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ 和 $b^{(0)},...,b^{(n_u-1)}$，其中模型把用户 $j$ 对电影 $i$ 的评分预测为 $y^{(i,j)} = \mathbf{w}^{(j)}\cdot \mathbf{x}^{(i)} + b^{(j)}$。给定一个由某些用户对某些电影给出的一组评分构成的数据集，你希望学习参数向量 $\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)}, \mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ 和 $b^{(0)},...,b^{(n_u-1)}$，使其产生最佳拟合（最小化平方误差）。

你将完成 cofiCostFunc 中的代码，以计算协同过滤的代价函数。


<a name="4.1"></a>
### 4.1 协同过滤代价函数

协同过滤代价函数由下式给出
$$J({\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},b^{(0)},...,\mathbf{w}^{(n_u-1)},b^{(n_u-1)}})= \left[ \frac{1}{2}\sum_{(i,j):r(i,j)=1}(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2 \right]
+ \underbrace{\left[
\frac{\lambda}{2}
\sum_{j=0}^{n_u-1}\sum_{k=0}^{n-1}(\mathbf{w}^{(j)}_k)^2
+ \frac{\lambda}{2}\sum_{i=0}^{n_m-1}\sum_{k=0}^{n-1}(\mathbf{x}_k^{(i)})^2
\right]}_{regularization}
\tag{1}$$
(1) 中的第一个求和是"对所有使得 $r(i,j)$ 等于 $1$ 的 $i$、$j$"，可以写成：

$$
= \left[ \frac{1}{2}\sum_{j=0}^{n_u-1} \sum_{i=0}^{n_m-1}r(i,j)*(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2 \right]
+\text{regularization}
$$

你现在应编写 cofiCostFunc（协同过滤代价函数）来返回这个代价。

<a name="ex01"></a>
### 练习 1

**for 循环实现：**
从用 for 循环实现代价函数开始。
考虑分两步来开发代价函数。首先，开发不含正则化的代价函数。下面提供了一个不含正则化的测试用例来测试你的实现。一旦它能工作，就加上正则化并运行包含正则化的测试。请注意，你只应在 $R(i,j) = 1$ 时为用户 $j$ 和电影 $i$ 累加代价。

In [4]:
# GRADED FUNCTION: cofi_cost_func
# UNQ_C1

def cofi_cost_func(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    nm, nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        for i in range(nm):
            J += np.square(R[i,j] * (np.dot(W[j,:],X[i,:]) + b[0,j] - Y[i,j]))        
    J = J / 2
    
    
    J =  J + ((lambda_/2) * (np.sum(np.square(W)) + np.sum(np.square(X))))
    
    ### END CODE HERE ### 

    return J

<details>
  <summary><font size="3" color="darkgreen"><b>点击获取提示</b></font></summary>
    你可以用两个 for 循环来组织代码，类似于 (1) 中的求和。
    先实现不含正则化的代码。
    请注意 (1) 中有些元素是向量。使用 np.dot()。你也可以使用 np.square()。
    要密切关注哪些元素由 i 索引、哪些由 j 索引。别忘了除以二。
    
```python     
    ### START CODE HERE ###  
    for j in range(nu):
        
        
        for i in range(nm):
            
            
    ### END CODE HERE ### 
```    
<details>
    <summary><font size="2" color="darkblue"><b> 点击获取更多提示</b></font></summary>
        
    这里有一些更多细节。下面的代码在使用每个元素之前先把它从矩阵中取出。
    你也可以直接引用矩阵。
    这段代码不含正则化。
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = 
            y = 
            r =
            J += 
    J = J/2
    ### END CODE HERE ### 

```
    
<details>
    <summary><font size="2" color="darkblue"><b>最后手段（完整的非正则化实现）</b></font></summary>
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += np.square(r * (np.dot(w,x) + b_j - y ) )
    J = J/2
    ### END CODE HERE ### 
```
    
<details>
    <summary><font size="2" color="darkblue"><b>正则化</b></font></summary>
     正则化只是把 W 数组和 X 数组的每个元素平方，然后把所有平方后的元素求和。
     你可以利用 np.square() 和 np.sum()。

<details>
    <summary><font size="2" color="darkblue"><b>正则化细节</b></font></summary>
    
```python 
    J += (lambda_/2) * (np.sum(np.square(W)) + np.sum(np.square(X)))
```
    
</details>
</details>
</details>
</details>

    

In [5]:
# Reduce the data set size so that this runs faster
num_users_r = 4
num_movies_r = 5 
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
b_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]

# Evaluate cost function
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

Cost: 13.67


**预期输出（lambda = 0）**：
$13.67$。

In [6]:
# Evaluate cost function with regularization 
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost (with regularization): 28.09


**预期输出**：

28.09

In [7]:
# Public tests
from public_tests import *
test_cofi_cost_func(cofi_cost_func)

All tests passed!


**向量化实现**

创建一个向量化的实现来计算 $J$ 是很重要的，因为它之后会在优化过程中被调用许多次。所用的线性代数不是本系列的重点，所以这里提供了实现。如果你是线性代数专家，可以随意在不参考下面代码的情况下创建你自己的版本。

运行下面的代码，并验证它产生与非向量化版本相同的结果。

In [8]:
def cofi_cost_func_v(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Vectorized for speed. Uses tensorflow operations to be compatible with custom training loop.
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y)*R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
    return J

In [9]:
# Evaluate cost function
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

# Evaluate cost function with regularization 
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost: 13.67
Cost (with regularization): 28.09


**预期输出**：
Cost: 13.67  
Cost (with regularization): 28.09

<a name="5"></a>
## 5 - 学习电影推荐 <img align="left" src="./images/film_man_action.png" style=" width:40px;  " >
------------------------------

在你完成协同过滤代价函数的实现之后，你就可以开始训练你的算法，为你自己做电影推荐了。

在下面的单元格中，你可以输入你自己的电影选择。然后算法将为你推荐！我们已经根据我们的偏好填写了一些值，但在你用我们的选择跑通之后，你应该把它改成符合你口味的。
数据集中所有电影的列表在文件 [movie list](data/small_movie_list.csv) 中。

In [10]:
movieList, movieList_df = load_Movie_List_pd()

my_ratings = np.zeros(num_movies)          #  Initialize my ratings

# Check the file small_movie_list.csv for id of each movie in our dataset
# For example, Toy Story 3 (2010) has ID 2700, so to rate it "5", you can set
my_ratings[2700] = 5 

#Or suppose you did not enjoy Persuasion (2007), you can set
my_ratings[2609] = 2;

# We have selected a few movies we liked / did not like and the ratings we
# gave are as follows:
my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)
my_rated = [i for i in range(len(my_ratings)) if my_ratings[i] > 0]

print('\nNew user ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');


New user ratings:

Rated 5.0 for  Shrek (2001)
Rated 5.0 for  Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Rated 2.0 for  Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Rated 5.0 for  Harry Potter and the Chamber of Secrets (2002)
Rated 5.0 for  Pirates of the Caribbean: The Curse of the Black Pearl (2003)
Rated 5.0 for  Lord of the Rings: The Return of the King, The (2003)
Rated 3.0 for  Eternal Sunshine of the Spotless Mind (2004)
Rated 5.0 for  Incredibles, The (2004)
Rated 2.0 for  Persuasion (2007)
Rated 5.0 for  Toy Story 3 (2010)
Rated 3.0 for  Inception (2010)
Rated 1.0 for  Louis Theroux: Law & Disorder (2008)
Rated 1.0 for  Nothing to Declare (Rien à déclarer) (2010)


现在，让我们把这些评论添加到 $Y$ 和 $R$ 中，并对评分进行归一化。

In [11]:
# Reload ratings
Y, R = load_ratings_small()

# Add new user ratings to Y 
Y = np.c_[my_ratings, Y]

# Add new user indicator matrix to R
R = np.c_[(my_ratings != 0).astype(int), R]

# Normalize the Dataset
Ynorm, Ymean = normalizeRatings(Y, R)

让我们准备训练模型。初始化参数并选择 Adam 优化器。

In [12]:
#  Useful Values
num_movies, num_users = Y.shape
num_features = 100

# Set Initial Parameters (W, X), use tf.Variable to track these variables
tf.random.set_seed(1234) # for consistent results
W = tf.Variable(tf.random.normal((num_users,  num_features),dtype=tf.float64),  name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features),dtype=tf.float64),  name='X')
b = tf.Variable(tf.random.normal((1,          num_users),   dtype=tf.float64),  name='b')

# Instantiate an optimizer.
optimizer = keras.optimizers.Adam(learning_rate=1e-1)

现在让我们训练协同过滤模型。这将学习参数 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$。

同时学习 $w$、$b$ 和 $x$ 所涉及的操作，并不属于 TensorFlow 神经网络软件包所提供的典型"层"。因此，课程 2 中使用的流程：Model、Compile()、Fit()、Predict()，不能直接适用。相反，我们可以使用一个自定义训练循环。

回顾前面实验中的梯度下降步骤。
- 重复直到收敛：
    - 计算前向传播
    - 计算损失相对于参数的导数
    - 使用学习率和计算出的导数来更新参数

TensorFlow 有一个很棒的能力，就是为你计算导数。这在下面有所展示。在 `tf.GradientTape()` 部分内，对 Tensorflow 变量的操作会被跟踪。当之后调用 `tape.gradient()` 时，它会返回损失相对于被跟踪变量的梯度。然后这些梯度可以用一个优化器应用到参数上。
这是对 TensorFlow 及其他机器学习框架一个有用特性的非常简短的介绍。可以通过研究你感兴趣框架内的"自定义训练循环"来找到更多信息。
    

In [13]:
iterations = 200
lambda_ = 1
for iter in range(iterations):
    # Use TensorFlow’s GradientTape
    # to record the operations used to compute the cost 
    with tf.GradientTape() as tape:

        # Compute the cost (forward pass included in cost)
        cost_value = cofi_cost_func_v(X, W, b, Ynorm, R, lambda_)

    # Use the gradient tape to automatically retrieve
    # the gradients of the trainable variables with respect to the loss
    grads = tape.gradient( cost_value, [X,W,b] )

    # Run one step of gradient descent by updating
    # the value of the variables to minimize the loss.
    optimizer.apply_gradients( zip(grads, [X,W,b]) )

    # Log periodically.
    if iter % 20 == 0:
        print(f"Training loss at iteration {iter}: {cost_value:0.1f}")

Training loss at iteration 0: 2321191.3
Training loss at iteration 20: 136169.3
Training loss at iteration 40: 51863.7
Training loss at iteration 60: 24599.0
Training loss at iteration 80: 13630.6
Training loss at iteration 100: 8487.7
Training loss at iteration 120: 5807.8
Training loss at iteration 140: 4311.6
Training loss at iteration 160: 3435.3
Training loss at iteration 180: 2902.1


<a name="6"></a>
## 6 - 推荐
在下面，我们为所有电影和用户计算评分，并显示被推荐的电影。这些是基于上面作为 `my_ratings[]` 输入的电影和评分。要预测用户 $j$ 对电影 $i$ 的评分，你计算 $\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)}$。这可以用矩阵乘法为所有评分计算出来。

In [14]:
# Make a prediction using trained weights and biases
p = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy()

#restore the mean
pm = p + Ymean

my_predictions = pm[:,0]

# sort predictions
ix = tf.argsort(my_predictions, direction='DESCENDING')

for i in range(17):
    j = ix[i]
    if j not in my_rated:
        print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

Predicting rating 4.49 for movie My Sassy Girl (Yeopgijeogin geunyeo) (2001)
Predicting rating 4.48 for movie Martin Lawrence Live: Runteldat (2002)
Predicting rating 4.48 for movie Memento (2000)
Predicting rating 4.47 for movie Delirium (2014)
Predicting rating 4.47 for movie Laggies (2014)
Predicting rating 4.47 for movie One I Love, The (2014)
Predicting rating 4.46 for movie Particle Fever (2013)
Predicting rating 4.45 for movie Eichmann (2007)
Predicting rating 4.45 for movie Battle Royale 2: Requiem (Batoru rowaiaru II: Chinkonka) (2003)
Predicting rating 4.45 for movie Into the Abyss (2011)


Original vs Predicted ratings:

Original 5.0, Predicted 4.90 for Shrek (2001)
Original 5.0, Predicted 4.84 for Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Original 2.0, Predicted 2.13 for Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Original 5.0, Predicted 4.88 for Harry Potter and the Chamber of Secrets (2002)
Original 5.0, Predic

在实践中，可以利用额外信息来增强我们的预测。在上面，前几百部电影的预测评分落在一个很小的范围内。我们可以通过从那些排名靠前的电影中，选出平均评分高的电影和评分数超过 20 的电影，来对上面的结果加以扩充。本节使用了一个 [Pandas](https://pandas.pydata.org/) 数据框，它有许多方便的排序特性。

In [15]:
filter=(movieList_df["number of ratings"] > 20)
movieList_df["pred"] = my_predictions
movieList_df = movieList_df.reindex(columns=["pred", "mean rating", "number of ratings", "title"])
movieList_df.loc[ix[:300]].loc[filter].sort_values("mean rating", ascending=False)

,pred,mean rating,number of ratings,title
1743,4.030961,4.252336,107,"Departed, The (2006)"
2112,3.985281,4.238255,149,"Dark Knight, The (2008)"
211,4.477798,4.122642,159,Memento (2000)
929,4.887054,4.118919,185,"Lord of the Rings: The Return of the King, The..."
2700,4.796531,4.109091,55,Toy Story 3 (2010)
653,4.357304,4.021277,188,"Lord of the Rings: The Two Towers, The (2002)"
1122,4.004471,4.006494,77,Shaun of the Dead (2004)
1841,3.980649,4.000000,61,Hot Fuzz (2007)
3083,4.084643,3.993421,76,"Dark Knight Rises, The (2012)"
2804,4.434171,3.989362,47,Harry Potter and the Deathly Hallows: Part 1 (...


<a name="7"></a>
## 7 - 恭喜完成！ <img align="left" src="./images/film_award.png"     style=" width:40px;  " >
你已经实现了一个有用的推荐系统！

<details>
  <summary><font size="2" color="darkgreen"><b>如果你想试验任何不参与评分的代码，请点击这里。</b></font></summary>
    <p><i><b>重要提示：请仅在你已经通过本次作业之后再这样做，以避免自动评分器出现问题。</b></i>
    <ol>
        <li> 在笔记本的菜单上，点击 "View" > "Cell Toolbar" > "Edit Metadata"</li>
        <li> 点击你想锁定/解锁的代码单元格旁边的 "Edit Metadata" 按钮</li>
        <li> 把 "editable" 的属性值设为：
            <ul>
                <li> 如果你想解锁它，设为 "true" </li>
                <li> 如果你想锁定它，设为 "false" </li>
            </ul>
        </li>
        <li> 在笔记本的菜单上，点击 "View" > "Cell Toolbar" > "None" </li>
    </ol>
    <p> 这里有一个简短的演示，展示如何执行上述步骤：
        <br>
        <img src="https://lh3.google.com/u/0/d/14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" align="center" alt="unlock_cells.gif">
</details>